# Cayley-Dickson Construction
This project is to represent the Cayley-Dickson construction for arbitrary base fields. <br>
When the base field is $\mathbb{R}$, this leads to the complex numbers, quaternions, octonions, etc.


In [2]:
import numpy as np

In [3]:
class Cayley:
    ''' A class to encode an array of numbers into a Cayley number.
        A CD_Alg object has the following attributes:
        coeff -  the array of coefficients as a numpy array. 
                
        field - the field of the coefficients
                'R' = real numbers, 0 = integers, p (prime)= finite field. 
                
        degree - the number of successive quadratic extensions needed to 
                represent the number. This is the smallest integer such 
                that 2^degree >= len(coeff). For example, the number 1
                has degree 0, the number 1 + i has degree 1, the number
                1 + i + j has degree 2, and so on. 
                
        real - the real part of the Cayley number.   '''
    
    def __init__(self, coeff=np.array([1]), degree=None, field='R'):
        if len(coeff) == 0: # edge case for empty array
            degree = 0
        if degree == None: # if degree not specified, use the smallest possible
            degree = (len(coeff) - 1).bit_length() # smallest extension needed
        if len(coeff) > (1 << degree): # check that it is large enough
            raise ValueError('Degree too small for number of coefficients')
        # pad with zeros
        coeff = np.pad(coeff, (0, (1 << degree) - len(coeff)), 'constant')

        if type(field) == int and field > 0: # reduce coefficients mod p
            p = field
            reduce = lambda x: x % p - p if x % p > p // 2 else x % p 
            self.coeff = np.array([reduce(coeff) for coeff in coeff])
        else:
            self.coeff = np.array(coeff)
        self.field = field
        self.degree = degree
        self.real = coeff[0]

    def __repr__(self):
        ''' written extremely badly! Definitely want to refactor this 
            and improve readability, but it works for now. '''
        if type(self.field) == int and self.field > 0: # reduce coefficients mod p
            p = self.field
            reduce = lambda x: x % p - p if x % p > p // 2 else x % p 
            self_disp_coeff = np.array([reduce(coeff) for coeff in self.coeff])
            return Cayley(self_disp_coeff, field=0).__repr__()

        def subscript(n):                
            # Base character 'i' in subscript Unicode
            base_char = chr(0x1D48A)
            
            # Unicode subscripts for digits 0-9
            subscripts = {
                '0': chr(0x2080),
                '1': chr(0x2081),
                '2': chr(0x2082),
                '3': chr(0x2083),
                '4': chr(0x2084),
                '5': chr(0x2085),
                '6': chr(0x2086),
                '7': chr(0x2087),
                '8': chr(0x2088),
                '9': chr(0x2089),
            }
            
            # Convert n into its subscript form
            subscript_digits = ''.join(subscripts[digit] for digit in str(n))
            
            # Return the result
            return base_char + subscript_digits
        
        def display(n):
            ''' Display the coefficient in a readable form '''
            if n == 0:
                return ''
            elif n == 1:
                return ' + '
            elif n == -1:
                return ' - '
            elif n < 0:
                return f' - {-1 * n}'
            else:
                return f' + {n}'
        
        non_zero_index = 0
        while self.coeff[non_zero_index] == 0:
            non_zero_index += 1
            if non_zero_index == len(self.coeff):
                return '0'
        if non_zero_index == 0:
            string = f'{self.coeff[non_zero_index]}' 
        else: 
            first_coeff = self.coeff[non_zero_index]
            if first_coeff == 1:
                first_coeff = ''
            elif first_coeff == -1:
                first_coeff = '-'
            string = f'{first_coeff}{subscript(non_zero_index)}'
        for digit in range(non_zero_index + 1, 1 << self.degree):
            if self.coeff[digit] != 0:
                string += f'{display(self.coeff[digit])}{subscript(digit)}'
        return string
    
    def left(self): # left half of coefficients as a Cayley number
        if self.degree == 0: # base case
            return self
        left_coeff = self.coeff[:1 << (self.degree - 1)]
        left_degree = self.degree - 1
        return Cayley(left_coeff, left_degree, self.field)
    
    def right(self): # right half of coefficients as a Cayley number
        if self.degree == 0: # base case
            return Cayley([0], 0, self.field)
        right_coeff = self.coeff[1 << (self.degree - 1):]
        right_degree = self.degree - 1
        return Cayley(right_coeff, right_degree, self.field)

    def conj(self): # conjugate of a Cayley number (a, b)* = (a*, -b)
        if self.degree == 0: # base case
            return self
        else:
            if self.field in ['R', 'Z', 0]: # characteristic 0 case
                conj_coeff = self.coeff.copy() * -1 
                conj_coeff[0] *= -1
            else: # characteristic p case
                p = self.field
                conj_coeff = (self.coeff.copy() * -1) % p
                conj_coeff[0] = (conj_coeff[0] * -1) % p
            return Cayley(conj_coeff, self.degree, self.field)

    def __eq__(self, other):
        D = max(self.degree, other.degree) # make sure the degrees match
        padded_self = Cayley(self.coeff, D, self.field)
        padded_other = Cayley(other.coeff, D, other.field)
        eq_coeffs = (list(padded_self.coeff) == list(padded_self.coeff))
        eq_field = (padded_self.field == padded_self.field)
        return eq_coeffs and eq_field
    
    def __add__(self, other):
        if self.field != other.field:
            raise ValueError('Fields must match')
        D = max(self.degree, other.degree) # make sure the degrees match
        padded_self = Cayley(self.coeff, D, self.field)
        padded_other = Cayley(other.coeff, D, other.field)
        sum_coeff = np.add(padded_self.coeff, padded_other.coeff)
        return Cayley(sum_coeff, D, self.field)
    
    def __sub__(self, other):
        if self.field != other.field:
            raise ValueError('Fields must match')
        D = max(self.degree, other.degree) # make sure the degrees match
        padded_self = Cayley(self.coeff, D, self.field)
        padded_other = Cayley(other.coeff, D, other.field)
        diff_coeff = np.subtract(padded_self.coeff, padded_other.coeff)
        return Cayley(diff_coeff, D, self.field)
    
    def __mul__(self, other):
        # Field dependent multiplication 
        def mult(a, b, field='R'):
            if field in ['R', 'Q', 'Z', 0]:
                return a * b
            elif type(field) == int:
                p = field # should be prime, but not checked
                return (a * b) % p
            else:
                raise ValueError(f'{field} is not a valid field')
            

        if type(other) in [float, int, np.int32, np.float64]: # Scalar 
            # if type(other) == int and self.field not in ['R', 0]:
            #     p = self.field  # characteristic p case
            #     return Cayley((self.coeff * other) % p, field=self.field)
            scaled_coeff = np.array([mult(other, coeff, self.field) 
                                     for coeff in self.coeff])
            return Cayley(scaled_coeff, field=self.field) # characteristic 0
        
        elif type(other) == Cayley: # Cayley number multiplication
            if self.field != other.field: # base fields should match
                raise ValueError('Fields must match to multiply')
            if self.degree == 0:    # base case is when the degree is 0
                return Cayley(other.coeff * self.coeff[0],
                                other.degree, self.field)
            elif other.degree == 0: 
                return Cayley(self.coeff * other.coeff[0],
                               self.degree, self.field)
            else: # break into smaller Cayley numbers for recursive multiplication
                D = max(self.degree, other.degree) # make sure the degrees match
                padded_self = Cayley(self.coeff, D, self.field)
                padded_other = Cayley(other.coeff, D, other.field)
                # multiplication formula is given by
                # (a_L, a_R)(b_L, b_R) = (a_L b_L - b_R* a_R, b_R a_L + a_R b_L* )
                a_L = Cayley(padded_self.left().coeff, D - 1, self.field)
                a_R = Cayley(padded_self.right().coeff, D - 1, self.field)
                b_L = Cayley(padded_other.left().coeff, D - 1, other.field)
                b_R = Cayley(padded_other.right().coeff, D - 1, other.field)

                prod_right = b_R * a_L + a_R * b_L.conj()  # recursive call 
                prod_left = a_L * b_L - b_R.conj() * a_R        
                product_coeff = np.concatenate((prod_left.coeff, prod_right.coeff))
                return Cayley(product_coeff, D, self.field)
        else:
            raise ValueError('Multiplication not defined for these types')
    
    def norm(self, as_scalar=True):
        ''' Returns the norm of a Cayley number N(x) = x.conj() * x'''
        if as_scalar:
            if type(self.field) == int and self.field > 0:
                p = self.field
                return np.sum((self.coeff ** 2) % p) % p
            return np.sum((self.coeff ** 2))
        else:
            return Cayley([self.norm(as_scalar=True)], 0, self.field)
    
    def inv(self):
        ''' Returns the inverse of a Cayley number, if it exists. '''
        if self.norm() == 0:
            raise ValueError('Cannot invert a Cayley number with norm 0')
        else:
            return self.conj() * (1 / self.norm())

In [4]:
a =Cayley([0.1,1,0,-1,0,-3,-.1,0,0,1], degree=None, field='R')
def imag(n): # returns the cayley number with only i_n component
    length = n.bit_length()
    coeff = np.zeros(1 << length, dtype='int32')
    coeff[n] = 1
    return Cayley(coeff, length, field=0)

A = np.zeros((8,8), dtype='object')
for i in range(8):
    for j in range(8):
        A[i,j] = str(imag(i) * imag(j))

A

array([['1', '𝒊₁', '𝒊₂', '𝒊₃', '𝒊₄', '𝒊₅', '𝒊₆', '𝒊₇'],
       ['𝒊₁', '-1', '𝒊₃', '-𝒊₂', '𝒊₅', '-𝒊₄', '-𝒊₇', '𝒊₆'],
       ['𝒊₂', '-𝒊₃', '-1', '𝒊₁', '𝒊₆', '𝒊₇', '-𝒊₄', '-𝒊₅'],
       ['𝒊₃', '𝒊₂', '-𝒊₁', '-1', '𝒊₇', '-𝒊₆', '𝒊₅', '-𝒊₄'],
       ['𝒊₄', '-𝒊₅', '-𝒊₆', '-𝒊₇', '-1', '𝒊₁', '𝒊₂', '𝒊₃'],
       ['𝒊₅', '𝒊₄', '-𝒊₇', '𝒊₆', '-𝒊₁', '-1', '-𝒊₃', '𝒊₂'],
       ['𝒊₆', '𝒊₇', '𝒊₄', '-𝒊₅', '-𝒊₂', '𝒊₃', '-1', '-𝒊₁'],
       ['𝒊₇', '-𝒊₆', '𝒊₅', '𝒊₄', '-𝒊₃', '-𝒊₂', '𝒊₁', '-1']], dtype=object)

In [12]:
for p in range(2,10):
    def base_b(n, b):
        ''' Returns the base b representation of n as a list of integers '''
        if n == 0:
            return [0]
        digits = []
        while n:
            digits.append(int(n % b))
            n //= b
        return digits
    def Cayley_p(n, p):
        return Cayley(base_b(n, p), field=p)

    norm_zeros = 0
    for i in range(p ** 4):
        if Cayley_p(i, p).norm() == 0:
            norm_zeros += 1
    print(p, norm_zeros, p ** 4 - norm_zeros, norm_zeros / (p ** 4))


2 8 8 0.5
3 33 48 0.4074074074074074
4 32 224 0.125
5 145 480 0.232
6 264 1032 0.2037037037037037
7 385 2016 0.16034985422740525
8 128 3968 0.03125
9 945 5616 0.1440329218106996


In [6]:
a = np.array([1,2,3,4])
np.sum(a) % 2

0

In [7]:
# idea for faster multiplication
def sgn(i, j):
    ''' e_ie_j = (-1) ^ sgn(i, j) e_(i XOR j) , where i > j'''
    if i == 0:
        return 0
    elif j == 0:
        return 0
    elif i == j:
        return 1
    elif j < i:
        return 1 - sgn(j, i)
    else:
        i_length = i.bit_length() # i = sum_{k < i_length} i_k 2^k

        hamming_i = i.bit_count() # |\{i_k : i_k = 1\}|
        i_bits = {k : (i & (1 << k)) >> k for k in range(i_length)} # {k : i_k}

        upper_j = j >> i_length # bits of j above biggest bit of i
        hamming_upper_j = upper_j.bit_count()
        lower_j_bits = {l : (j & (1 << l)) >> l for l in range(i_length)} 

        # if j is a power of 2 bigger than i, all upper terms of j will count
        # with all lower terms of i, so it's faster to multiply them directly
        result = (hamming_i % 2) * (hamming_upper_j % 2) 
        high_inx = 0
        while high_inx in lower_j_bits: # add i_k * j_l for all k <= l
            for low_inx in range(high_inx + 1):
                if i_bits[low_inx] == 1 and lower_j_bits[high_inx] == 1:
                    result = 1 - result
            high_inx += 1
        return 1 - result

def fast_mult(cayley1, cayley2):
    ''' multiplication on basis using sgn and extend by distributive law'''
    D = max(cayley1.degree, cayley2.degree)
    prod_coeff = np.zeros(1 << D, dtype='int32')
    for i in range(1 << cayley1.degree):
        for j in range(1 << cayley2.degree):
            sign = 1 - 2 * (sgn(i, j) % 2) # (-1) ^ sgn(i, j)
            prod_coeff[i ^ j] += sign * cayley1.coeff[i] * cayley2.coeff[j]
    return Cayley(prod_coeff, D, cayley1.field)


In [11]:
i = Cayley([0,1], degree=1, field='R')
j = Cayley([0,0,1,0], degree=2, field='R')
k = Cayley([0,0,0,1])
x = Cayley([n for n in range(1 << 6)])
y = Cayley([ n**2 for n in range(1 << 6)])
print(x * y == fast_mult(x,y))

%time x * y
%time fast_mult(x,y)

True
CPU times: total: 688 ms
Wall time: 692 ms
CPU times: total: 31.2 ms
Wall time: 25.9 ms


-4064256 - 15624𝒊₁ + 12824𝒊₂ - 24088𝒊₃ + 53424𝒊₄ + 43448𝒊₅ - 44024𝒊₆ + 41936𝒊₇ - 77216𝒊₈ + 75448𝒊₉ + 94072𝒊₁₀ + 79840𝒊₁₁ - 72976𝒊₁₂ + 100168𝒊₁₃ + 78248𝒊₁₄ + 101280𝒊₁₅ - 68544𝒊₁₆ - 183800𝒊₁₇ + 64808𝒊₁₈ - 180848𝒊₁₉ + 191120𝒊₂₀ + 76632𝒊₂₁ - 179720𝒊₂₂ + 68208𝒊₂₃ - 97952𝒊₂₄ + 198680𝒊₂₅ + 92856𝒊₂₆ + 198864𝒊₂₇ - 173168𝒊₂₈ + 78408𝒊₂₉ + 186792𝒊₃₀ + 87984𝒊₃₁ + 170624𝒊₃₂ + 243272𝒊₃₃ - 170584𝒊₃₄ + 227360𝒊₃₅ - 178064𝒊₃₆ - 177544𝒊₃₇ + 205240𝒊₃₈ - 168096𝒊₃₉ + 188320𝒊₄₀ - 98840𝒊₄₁ - 173752𝒊₄₂ - 113856𝒊₄₃ + 161552𝒊₄₄ - 152040𝒊₄₅ - 133224𝒊₄₆ - 156864𝒊₄₇ - 102208𝒊₄₈ + 184120𝒊₄₉ + 67928𝒊₅₀ + 177840𝒊₅₁ - 134032𝒊₅₂ + 28392𝒊₅₃ + 155016𝒊₅₄ + 46032𝒊₅₅ + 69664𝒊₅₆ - 67416𝒊₅₇ - 34488𝒊₅₈ - 71184𝒊₅₉ + 96240𝒊₆₀ + 8952𝒊₆₁ - 74664𝒊₆₂ - 6512𝒊₆₃

In [ ]:
print([f'{imag(i)} * {imag(j)} = {imag(i) * imag(j)}' for i in range(8) for j in range(8)])

['1 * 1 = 1', '1 * 𝒊₁ = 𝒊₁', '1 * 𝒊₂ = 𝒊₂', '1 * 𝒊₃ = 𝒊₃', '1 * 𝒊₄ = 𝒊₄', '1 * 𝒊₅ = 𝒊₅', '1 * 𝒊₆ = 𝒊₆', '1 * 𝒊₇ = 𝒊₇', '𝒊₁ * 1 = 𝒊₁', '𝒊₁ * 𝒊₁ = -1', '𝒊₁ * 𝒊₂ = 𝒊₃', '𝒊₁ * 𝒊₃ = -𝒊₂', '𝒊₁ * 𝒊₄ = 𝒊₅', '𝒊₁ * 𝒊₅ = -𝒊₄', '𝒊₁ * 𝒊₆ = -𝒊₇', '𝒊₁ * 𝒊₇ = 𝒊₆', '𝒊₂ * 1 = 𝒊₂', '𝒊₂ * 𝒊₁ = -𝒊₃', '𝒊₂ * 𝒊₂ = -1', '𝒊₂ * 𝒊₃ = 𝒊₁', '𝒊₂ * 𝒊₄ = 𝒊₆', '𝒊₂ * 𝒊₅ = 𝒊₇', '𝒊₂ * 𝒊₆ = -𝒊₄', '𝒊₂ * 𝒊₇ = -𝒊₅', '𝒊₃ * 1 = 𝒊₃', '𝒊₃ * 𝒊₁ = 𝒊₂', '𝒊₃ * 𝒊₂ = -𝒊₁', '𝒊₃ * 𝒊₃ = -1', '𝒊₃ * 𝒊₄ = 𝒊₇', '𝒊₃ * 𝒊₅ = -𝒊₆', '𝒊₃ * 𝒊₆ = 𝒊₅', '𝒊₃ * 𝒊₇ = -𝒊₄', '𝒊₄ * 1 = 𝒊₄', '𝒊₄ * 𝒊₁ = -𝒊₅', '𝒊₄ * 𝒊₂ = -𝒊₆', '𝒊₄ * 𝒊₃ = -𝒊₇', '𝒊₄ * 𝒊₄ = -1', '𝒊₄ * 𝒊₅ = 𝒊₁', '𝒊₄ * 𝒊₆ = 𝒊₂', '𝒊₄ * 𝒊₇ = 𝒊₃', '𝒊₅ * 1 = 𝒊₅', '𝒊₅ * 𝒊₁ = 𝒊₄', '𝒊₅ * 𝒊₂ = -𝒊₇', '𝒊₅ * 𝒊₃ = 𝒊₆', '𝒊₅ * 𝒊₄ = -𝒊₁', '𝒊₅ * 𝒊₅ = -1', '𝒊₅ * 𝒊₆ = -𝒊₃', '𝒊₅ * 𝒊₇ = 𝒊₂', '𝒊₆ * 1 = 𝒊₆', '𝒊₆ * 𝒊₁ = 𝒊₇', '𝒊₆ * 𝒊₂ = 𝒊₄', '𝒊₆ * 𝒊₃ = -𝒊₅', '𝒊₆ * 𝒊₄ = -𝒊₂', '𝒊₆ * 𝒊₅ = 𝒊₃', '𝒊₆ * 𝒊₆ = -1', '𝒊₆ * 𝒊₇ = -𝒊₁', '𝒊₇ * 1 = 𝒊₇', '𝒊₇ * 𝒊₁ = -𝒊₆', '𝒊₇ * 𝒊₂ = 𝒊₅', '𝒊₇ * 𝒊₃ = 𝒊₄', '𝒊₇ * 𝒊₄ = -𝒊₃', '𝒊₇ * 𝒊₅ = -𝒊₂', '𝒊₇